# Model Context Protocol (MCP) — Demo pratica

## Cos'è MCP?

**MCP (Model Context Protocol)** è uno standard aperto creato da Anthropic per connettere LLM a tool e dati esterni in modo **uniforme e interoperabile**.

```
Prima di MCP:          Con MCP:

App A ─── Tool 1      App A ╮
App B ─── Tool 1      App B ├── MCP Server ── Tool 1
App C ─── Tool 1      App C ╯
  (ogni app riscrive tutto)    (un server, molti client)
```

**Analogia:** MCP sta agli LLM come USB sta ai dispositivi — un connettore universale.

### Componenti
| Componente | Ruolo |
|------------|-------|
| **MCP Server** | Espone tool (e risorse) via JSON-RPC |
| **MCP Client** | Si connette al server, lista e chiama tool |
| **Transport** | Come comunicano: `stdio` (subprocess) o `SSE` (HTTP) |

In questa demo usiamo **stdio**: il client avvia il server come subprocess e comunica via stdin/stdout. Nessun server HTTP da gestire!

## 1. Setup

Le dipendenze sono già installate nell'ambiente `uv` del progetto. Se parti da zero:

In [ ]:
# Decommentare solo se NON si usa l'ambiente uv del progetto
# %pip install "mcp[cli]" smolagents pytz python-dotenv

In [1]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Client MCP
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.session import ClientSession

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Percorso assoluto al server (necessario per StdioServerParameters)
SERVER_PATH = str(Path("mcp_server.py").resolve())

print("Import OK")
print(f"Server: {SERVER_PATH}")

Import OK
Server: /Users/giovannibonetta/Desktop/progetti/agent_demo/mcp_server.py


## 2. Il Server MCP

Vediamo il codice del server prima di avviarlo. Si trova nel file [mcp_server.py](mcp_server.py).

Tre cose importanti:
1. `FastMCP` gestisce tutto il protocollo — noi scriviamo solo funzioni Python
2. Il decoratore `@mcp.tool()` registra la funzione come tool MCP
3. Il docstring **è il contratto**: descrive al client (e all'LLM) cosa fa il tool e cosa si aspetta

In [2]:
# Leggiamo e stampiamo il codice del server per esaminarlo insieme
print(Path("mcp_server.py").read_text())

"""
MCP Server di Demo — usato dal notebook mcp_demo.ipynb

Espone 4 tool via Model Context Protocol (trasporto stdio).
Avviato automaticamente dal client come subprocess; non serve
aprire un secondo terminale.
"""

from mcp.server.fastmcp import FastMCP
import random
import datetime
import pytz

# FastMCP gestisce l'intero protocollo MCP: registrazione tool,
# serializzazione JSON-RPC e trasporto stdio.
mcp = FastMCP("DemoServer")


# ── Tool 1 ──────────────────────────────────────────────────────────────────
@mcp.tool()
def count_words(text: str) -> int:
    """Conta il numero di parole in un testo.

    Args:
        text: Il testo da analizzare.
    """
    return len(text.split())


# ── Tool 2 ──────────────────────────────────────────────────────────────────
@mcp.tool()
def convert_temperature(value: float, from_unit: str, to_unit: str) -> str:
    """Converte una temperatura tra Celsius (C), Fahrenheit (F) e Kelvin (K).

    Args:
        value: Il valore numerico della temper

## 3. Connettersi al Server — Client Puro

Il client MCP usa `asyncio`. In Jupyter possiamo usare `await` direttamente senza `asyncio.run()`.

### 3a. Listare i Tool disponibili

`session.list_tools()` chiede al server quali tool espone — il server risponde con nome, descrizione e schema degli argomenti.

In [3]:
# StdioServerParameters dice al client come avviare il server:
# "esegui `python mcp_server.py` come subprocess"
server_params = StdioServerParameters(
    command=sys.executable,   # usa l'interprete Python del kernel attivo
    args=[SERVER_PATH],
)

# Apriamo la connessione e listiamo i tool
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        tools_response = await session.list_tools()

        print(f"Tool esposti dal server: {len(tools_response.tools)}\n")
        for tool in tools_response.tools:
            print(f"  [{tool.name}]")
            print(f"    {tool.description}")
            argomenti = list(tool.inputSchema.get("properties", {}).keys())
            print(f"    Argomenti: {argomenti}")
            print()

Tool esposti dal server: 4

  [count_words]
    Conta il numero di parole in un testo.

    Args:
        text: Il testo da analizzare.
    
    Argomenti: ['text']

  [convert_temperature]
    Converte una temperatura tra Celsius (C), Fahrenheit (F) e Kelvin (K).

    Args:
        value: Il valore numerico della temperatura.
        from_unit: Unità di partenza: 'C', 'F' o 'K'.
        to_unit: Unità di arrivo: 'C', 'F' o 'K'.
    
    Argomenti: ['value', 'from_unit', 'to_unit']

  [get_current_time]
    Restituisce l'ora corrente in un fuso orario specificato.

    Args:
        timezone: Fuso orario IANA valido, es. 'Europe/Rome', 'America/New_York'.
    
    Argomenti: ['timezone']

  [get_random_fact]
    Restituisce un fatto curioso casuale dalla collezione del server.
    Argomenti: []



### 3b. Chiamare Tool

Ogni chiamata a `session.call_tool(nome, argomenti)` è una richiesta JSON-RPC al server. Il server esegue la funzione e restituisce il risultato.

In [4]:
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # ── Tool 1: conta parole ─────────────────────────────────────────
        r1 = await session.call_tool(
            "count_words",
            {"text": "Il modello context protocol è uno standard aperto molto utile"}
        )
        print("count_words:", r1.content[0].text)

        # ── Tool 2: conversione temperatura ─────────────────────────────
        r2 = await session.call_tool(
            "convert_temperature",
            {"value": 37.0, "from_unit": "C", "to_unit": "F"}
        )
        print("convert_temperature:", r2.content[0].text)

        # ── Tool 3: ora corrente ─────────────────────────────────────────
        r3 = await session.call_tool(
            "get_current_time",
            {"timezone": "Europe/Rome"}
        )
        print("get_current_time:", r3.content[0].text)

        # ── Tool 4: fatto casuale ────────────────────────────────────────
        r4 = await session.call_tool("get_random_fact", {})
        print("get_random_fact:", r4.content[0].text)

count_words: 10
convert_temperature: 37.0°C = 98.60°F
get_current_time: Ora in Europe/Rome: 2026-05-10 21:46:36
get_random_fact: Le api possono riconoscere i volti umani.


## 4. MCP + smolagents — Integrazione con un Agente AI

Questa è la parte più interessante: colleghiamo il server MCP a un agente `smolagents`.

**`MCPClient`** è il ponte: converte automaticamente i tool MCP in tool compatibili con `CodeAgent`.  
L'agente non sa (e non deve sapere) che i tool arrivano da un server MCP — li usa come qualsiasi altro tool.

```
CodeAgent
    │
    ├── MCPClient ── stdio ── mcp_server.py ── [count_words, convert_temperature, ...]
    └── (altri tool locali se vuoi)
```

In [5]:
from smolagents import MCPClient, CodeAgent, InferenceClientModel

# Il modello LLM (stesso del notebook agent_demo)
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=HF_TOKEN,
    max_tokens=2048,
    temperature=0.5,
)

print("Modello pronto")

Modello pronto


In [6]:
# MCPClient si usa come context manager: apre la connessione e la chiude automaticamente
with MCPClient(server_params) as mcp_tools:

    print("Tool MCP disponibili per l'agente:")
    for t in mcp_tools:
        print(f"  - {t.name}")

    # Creiamo l'agente passando i tool MCP direttamente
    agent = CodeAgent(
        model=model,
        tools=mcp_tools,     # lista di tool dal server MCP
        max_steps=4,
        verbosity_level=1,
    )

    # L'agente usa i tool MCP come se fossero tool locali
    risposta = agent.run(
        "Converti 100 gradi Fahrenheit in Celsius e dimmi che ore sono a Tokyo."
    )

print("\n" + "="*50)
print("RISPOSTA:", risposta)

/var/folders/zq/qybh5byx75n386y34knx4crm0000gn/T/ipykernel_95890/335529244.py:2: FutureWarning: Parameter 'structured_output' was not specified. Currently it defaults to False, but in version 1.25, the default will change to True. To suppress this warning, explicitly set structured_output=True (new behavior) or structured_output=False (legacy behavior). See documentation at https://huggingface.co/docs/smolagents/tutorials/tools#structured-output-and-output-schema-support for more details.
  with MCPClient(server_params) as mcp_tools:


Tool MCP disponibili per l'agente:
  - count_words
  - convert_temperature
  - get_current_time
  - get_random_fact


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Converti 100 gradi Fahrenheit in Celsius e dimmi che ore sono a Tokyo.                                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Convertire 100 gradi Fahrenheit in Celsius                                                                     
  celsius = convert_temperature(value=100, from_unit="F", to_unit="C")                                             
  print(f"100 gradi Fahrenheit equivalgono a {celsius} gradi Celsius")                                             
                                                                                                                   
  # Ottenere l'ora corrente a Tokyo                                                                                
  time_tokyo = get_current_time(timezone="Asia/Tokyo")                                                             
  print(f"L'ora corrente a Tokyo è {time_tokyo}")                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"100 gradi Fahrenheit equivalgono a 37.78 gradi Celsius. L'ora corrente a Tokyo è 04:46:53 del 11  
  maggio 2026.")                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 100 gradi Fahrenheit equivalgono a 37.78 gradi Celsius. L'ora corrente a Tokyo è 04:46:53 del 11 
maggio 2026.

[Step 2: Duration 1.50 seconds| Input tokens: 5,015 | Output tokens: 278]


RISPOSTA: 100 gradi Fahrenheit equivalgono a 37.78 gradi Celsius. L'ora corrente a Tokyo è 04:46:53 del 11 maggio 2026.


In [ ]:
# Seconda query: l'agente usa più tool in sequenza
with MCPClient(server_params) as mcp_tools:
    agent = CodeAgent(
        model=model,
        tools=mcp_tools,
        max_steps=4,
        verbosity_level=1,
    )

    risposta = agent.run(
        "Conta le parole in questa frase: 'Il Model Context Protocol semplifica "
        "l integrazione degli LLM con strumenti esterni'. "
        "Poi dimmi un fatto curioso."
    )

print("\n" + "="*50)
print("RISPOSTA:", risposta)

## 5. Esercizio — Aggiungere un Tool al Server

Apri `mcp_server.py` e aggiungi un nuovo tool, ad esempio:
- `reverse_text(text)` — inverte una stringa
- `fibonacci(n)` — calcola l'n-esimo numero di Fibonacci
- `is_palindrome(word)` — controlla se una parola è palindroma

Poi riesegui le celle sopra: il client vedrà il nuovo tool senza modifiche al client stesso.

**Questo è il punto chiave di MCP**: il server e il client evolvono indipendentemente.

In [8]:
# Dopo aver modificato mcp_server.py, esegui questa cella per vedere il nuovo tool
async with stdio_client(server_params) as (r, w):
    async with ClientSession(r, w) as session:
        await session.initialize()
        tools = await session.list_tools()
        print("Tool disponibili ora:")
        for t in tools.tools:
            print(f"  - {t.name}: {t.description}")

Tool disponibili ora:
  - count_words: Conta il numero di parole in un testo.

    Args:
        text: Il testo da analizzare.
    
  - convert_temperature: Converte una temperatura tra Celsius (C), Fahrenheit (F) e Kelvin (K).

    Args:
        value: Il valore numerico della temperatura.
        from_unit: Unità di partenza: 'C', 'F' o 'K'.
        to_unit: Unità di arrivo: 'C', 'F' o 'K'.
    
  - get_current_time: Restituisce l'ora corrente in un fuso orario specificato.

    Args:
        timezone: Fuso orario IANA valido, es. 'Europe/Rome', 'America/New_York'.
    
  - get_random_fact: Restituisce un fatto curioso casuale dalla collezione del server.
  - reverse_string: Restituisce la stringa invertita.


## 6. Server MCP in Rete — Trasporto SSE e Streamable HTTP

Finora abbiamo usato il trasporto **stdio**: il client avvia il server come subprocess locale. Esistono altri due trasporti per esporre un server MCP via rete:

| Trasporto | Come funziona | Quando usarlo |
|-----------|---------------|---------------|
| `stdio` | subprocess locale, stdin/stdout | Tool locali, demo |
| `SSE` | HTTP + Server-Sent Events (stream unidirezionale) | Server legacy, compatibilità ampia |
| `streamable-http` | HTTP bidirezionale (nuovo standard MCP) | Server moderni, raccomandato |

Esistono già server MCP pubblici accessibili a tutti — non serve che ne deployamo uno noi.

In questa sezione usiamo:
- **Cloudflare Docs** (`SSE`) — documentazione Cloudflare, nessuna autenticazione
- **HuggingFace Hub** (`streamable-http`) — cerca modelli, dataset e paper, con HF Token

In [4]:
import httpx
from mcp.client.sse import sse_client
from mcp.client.streamable_http import streamable_http_client

# URL dei server pubblici che useremo in questa sezione
CLOUDFLARE_URL = "https://docs.mcp.cloudflare.com/sse"
HF_MCP_URL = "https://hf.co/mcp"

# Gli header di autenticazione vanno passati tramite httpx.AsyncClient
hf_http_client = httpx.AsyncClient(
    headers={"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
)

print("Import transport OK")
print(f"Cloudflare: {CLOUDFLARE_URL}")
print(f"HuggingFace Hub: {HF_MCP_URL}")

Import transport OK
Cloudflare: https://docs.mcp.cloudflare.com/sse
HuggingFace Hub: https://hf.co/mcp


### 6a. Cloudflare Docs — SSE, nessuna autenticazione

Con `sse_client` la connessione funziona esattamente come con `stdio_client`: stessa `ClientSession`, stesse chiamate `list_tools` e `call_tool`. Cambia solo il modo in cui si apre il canale di comunicazione.

In [5]:
async with sse_client(CLOUDFLARE_URL) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        tools = await session.list_tools()
        print(f"Tool esposti da Cloudflare Docs ({len(tools.tools)} totali):")
        for t in tools.tools:
            print(f"  [{t.name}]")
            print(f"    {t.description}")

Tool esposti da Cloudflare Docs (2 totali):
  [search_cloudflare_documentation]
    Search the Cloudflare documentation.

		This tool should be used to answer any question about Cloudflare products or features, including:
		- Workers, Pages, R2, Images, Stream, D1, Durable Objects, KV, Workflows, Hyperdrive, Queues
		- AI Search, Workers AI, Vectorize, AI Gateway, Browser Rendering
		- Zero Trust, Access, Tunnel, Gateway, Browser Isolation, WARP, DDOS, Magic Transit, Magic WAN
		- CDN, Cache, DNS, Zaraz, Argo, Rulesets, Terraform, Account and Billing

		Results are returned as semantically similar chunks to the query.
		
  [migrate_pages_to_workers_guide]
    ALWAYS read this guide before migrating Pages projects to Workers.


In [6]:
# Chiamiamo il tool di ricerca documentazione
async with sse_client(CLOUDFLARE_URL) as (r, w):
    async with ClientSession(r, w) as session:
        await session.initialize()

        result = await session.call_tool(
            "search_cloudflare_documentation",
            {"query": "AI workers inference", "limit": 3}
        )
        print(result.content[0].text[:600], "...")

<result>
<url>https://developers.cloudflare.com/https://developers.cloudflare.com/reference-architecture/diagrams/ai/ai-multivendor-observability-control/</url>
<title></title>
<text>
# Multi-vendor AI observability and control

**Last reviewed:**  almost 2 years ago 

## Introduction

The AI landscape is rapidly evolving with new models, services, and applications emerging daily. Many developers and organizations seek to enhance agility by opting for inference-as-a-service solutions like [Workers AI](/workers-ai/), rather than developing or managing models themselves.

Inference-as-a-Service  ...


### 6b. HuggingFace Hub — Streamable HTTP, con HF Token

Il server ufficiale HuggingFace espone tool per cercare nel Hub: modelli, dataset, Spaces e paper.

Con `streamablehttp_client` il context manager restituisce **tre** valori `(read, write, _)` invece di due (il terzo è la sessione HTTP sottostante, che ignoriamo con `_`).

Il token è opzionale ma aumenta i rate limit.

In [7]:
async with streamable_http_client(HF_MCP_URL, http_client=hf_http_client) as (r, w, _):
    async with ClientSession(r, w) as session:
        await session.initialize()

        tools = await session.list_tools()
        print(f"Tool esposti da HuggingFace Hub ({len(tools.tools)} totali):")
        for t in tools.tools:
            print(f"  [{t.name}] {t.description[:70]}")

Tool esposti da HuggingFace Hub (10 totali):
  [hf_whoami] Hugging Face tools are being used by authenticated user 'giobin'
  [space_search] Find Hugging Face Spaces using semantic search. IMPORTANT Only MCP Ser
  [hub_repo_search] Search Hugging Face repositories with a shared query interface. You ca
  [paper_search] Find Machine Learning research papers on the Hugging Face hub. Include
  [hub_repo_details] Get details for one or more Hugging Face repos (model, dataset, or spa
  [hf_doc_search] Search and Discover Hugging Face Product and Library documentation. Se
  [hf_doc_fetch] Fetch a document from the Hugging Face or Gradio documentation library
  [dynamic_space] Perform Tasks with Hugging Face Spaces. Use "discover" to view availab
  [hf_hub_query] Read-only Hugging Face Hub navigator for discovery, lookup, filtering,
  [gr1_z_image_turbo_generate] Generate an image using the Z-Image model based on the provided prompt


In [8]:
# Cerchiamo modelli e paper legati al corso che stiamo seguendo
async with streamable_http_client(HF_MCP_URL, http_client=hf_http_client) as (r, w, _):
    async with ClientSession(r, w) as session:
        await session.initialize()

        r1 = await session.call_tool(
            "hub_repo_search",
            {"query": "smolagents agent", "limit": 3}
        )
        print("=== Modelli 'smolagents agent' ===")
        print(r1.content[0].text[:600])

        print()

        r2 = await session.call_tool(
            "paper_search",
            {"query": "LLM tool use agents", "limit": 3}
        )
        print("=== Paper 'LLM tool use agents' ===")
        print(r2.content[0].text[:600])

=== Modelli 'smolagents agent' ===
Found 4 repositories across models, datasets matching query "smolagents agent".

## Models (3)

### smolagents/SmolVLM2-2.2B-Instruct-Agentic-GUI

**Task:** image-text-to-text | **Library:** transformers | **Downloads:** 78 | **Likes:** 65 | **Trending Score:** 1

**Tags:** transformers, safetensors, smolvlm, image-text-to-text, generated_from_trainer, open-r1, vision-language, vlm, conversational, dataset:smolagents/aguvis-stage-2, endpoints_compatible, region:us

**Created:** 28 Jul, 2025
**Link:** [https://hf.co/smolagents/SmolVLM2-2.2B-Instruct-Agentic-GUI](https://hf.co/smolagents/SmolVLM

=== Paper 'LLM tool use agents' ===
120 papers matched the query 'LLM tool use agents'. Here are the first 12 results.

---

## CostBench: Evaluating Multi-Turn Cost-Optimal Planning and Adaptation in
  Dynamic Environments for LLM Tool-Use Agents

Published on Published on 4 Nov, 2025
**Authors:** Jiayu Liu ([JiayuJeff](https://hf.co/JiayuJeff)), Cheng Qian, Z

### 6c. HuggingFace Hub + smolagents — Agente che cerca nel Hub

Per usare `MCPClient` con un server remoto, si passa un **dizionario** con `url` e `transport` al posto di `StdioServerParameters`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# MCPClient con server remoto: dict con url e transport
# Per l'autenticazione si passa headers nel dict (MCPAdapt gestisce lui httpx internamente)
hf_server_config = {
    "url": HF_MCP_URL,
    "transport": "streamable-http",
    "headers": {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {},
}

with MCPClient(hf_server_config, structured_output=False) as hf_tools:
    print("Tool HF disponibili per l'agente:")
    for t in hf_tools:
        print(f"  - {t.name}")

    agent = CodeAgent(
        model=model,
        tools=hf_tools,
        max_steps=4,
        verbosity_level=1,
    )

    risposta = agent.run(
        "Cerca i 3 modelli più popolari su HuggingFace Hub legati agli agenti AI "
        "e dimmi brevemente cosa fa ciascuno."
    )

print("\n" + "="*50)
print("RISPOSTA:", risposta)

## Riepilogo

| Concetto | Cosa abbiamo visto |
|----------|-------------------|
| **MCP Server** | `FastMCP` + `@mcp.tool()` — stessa semplicità di `@tool` in smolagents |
| **Trasporto stdio** | Il client avvia il server come subprocess: zero infrastruttura |
| **Trasporto SSE** | `sse_client` — server HTTP accessibile in rete, no auth (Cloudflare Docs) |
| **Trasporto streamable-http** | `streamablehttp_client` — nuovo standard, restituisce 3 valori `(r, w, _)` |
| **Client puro** | `ClientSession` — `list_tools` e `call_tool` funzionano uguali con tutti i trasporti |
| **Integrazione agente** | `MCPClient` accetta sia `StdioServerParameters` che `dict` per server remoti |
| **Separazione server/client** | Aggiungere tool al server non richiede modifiche al client |

### Custom tool vs MCP — quando usare cosa?

| Scenario | Approccio consigliato |
|----------|----------------------|
| Tool usa solo Python, per un solo agente | `@tool` direttamente in smolagents |
| Tool da condividere tra più agenti/app | MCP Server |
| Tool già esistente in un'altra app | MCP Server (esponi ciò che esiste già) |
| Tool con dipendenze pesanti (GPU, DB) | MCP Server (gira separato, non appesantisce il client) |
| Server pubblico di terze parti | `sse_client` o `streamablehttp_client` con URL remoto |